# YouTube Shorts Ideas from a Long-Form Video Transcript

Turn one long YouTube video into **5 Shorts ideas**. Paste a video URL, this notebook fetches the captions, and an LLM suggests clips you could cut — each idea includes a title, a short description, and start/end times from the original transcript.

**Learning / personal use only.** YouTube often blocks caption requests from cloud IPs (AWS, Azure, etc.), so this is meant to run locally, not as a commercial product.

## What you need before you start

1. **OpenAI API key** in your environment as `OPENAI_API_KEY` (same setup as the rest of this course).
2. A YouTube video that **has captions**.
3. A URL in this shape: `https://www.youtube.com/watch?v=VIDEO_ID`  
   (The code only reads the `v=` query parameter — not `youtu.be` or `/shorts/` links.)

## How to run

1. Run **every cell from top to bottom** (do not skip).
2. In **Step 3**, you can replace the sample `video_url` with your own `watch?v=` link, then re-run from that cell onward.
3. The last cell prints the ideas as formatted JSON.

## What each step does

| Step | What happens |
|------|----------------|
| 1 | Installs `youtube-transcript-api` |
| 2 | Loads OpenAI + helper functions to pull the transcript |
| 3 | Fetches captions for the URL you set |
| 4 | Builds the system and user prompts |
| 5 | Calls the model and displays 5 Shorts ideas |

If a cell fails, check: API key is set, the video has captions, and the URL contains `v=`.

## Step 1 — Install the YouTube transcript package

Run this once per environment. It installs `youtube-transcript-api`, which downloads the video’s captions. The OpenAI Python package is assumed to already be available from the course setup.


In [ ]:
%pip install youtube-transcript-api

## Step 2 — Imports and transcript helpers

This cell:

- Creates an OpenAI client from `OPENAI_API_KEY`
- Defines `get_video_id()` — takes the ID after `v=` in the URL
- Defines `get_transcript()` — fetches captions for that ID

Run it after Step 1 so the functions exist before you fetch a video.

In [ ]:
from openai import OpenAI
import os
import json
from IPython.display import Markdown
from youtube_transcript_api import YouTubeTranscriptApi

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_transcript(url):
    try:
        video_id = get_video_id(url)
        print(video_id)
        transcript = YouTubeTranscriptApi().fetch(video_id)
        return transcript
    except Exception as e:
        print(f"Error getting transcript for video {video_id}: {e}")
        return None


def get_video_id(url):
    video_id = url.split("v=")[1]
    return video_id



## Step 3 — Fetch the transcript

Set `video_url` to a captioned YouTube video, then run the cell. A sample URL is already filled in.

To try your own video: paste a `https://www.youtube.com/watch?v=...` link, run this cell again, then continue with Steps 4 and 5.

If this fails, the video may have no captions, or the URL may not include `v=`.

In [ ]:
video_url = "https://www.youtube.com/watch?v=qp0HIF3SfI4"
transcriptJSON = get_transcript(video_url).to_raw_data()

## Step 4 — System prompt and user prompt

The **system prompt** tells the model to return **5 Shorts ideas** as JSON. Each idea should include:

- `idea` — a short title
- `description` — what the Short would cover
- `original_transcript_start_time` / `original_transcript_end_time` — where that moment sits in the long video

The **user prompt** is the transcript from Step 3. Run this cell so both prompts are defined before the API call.

In [ ]:
system_prompt = """
You are a helpful assistant that can help me generate ideas for YouTube Shorts.
Given you get a youtube video transcript, you should carefully analyze the transcript and generate ideas for YouTube Shorts.
You should generate 5 ideas for YouTube Shorts based on the transcript.


The ideas should be unique and creative.
The ideas should be relevant to the transcript.
The ideas should be engaging and interesting to the audience.
The ideas should be easy to understand and implement.

The output should be a json array of strings, where each string is an idea for a YouTube Short.
`[{
    "idea": "string",
    "description": "string",
    "original_transcript_start_time": "number",
    "original_transcript_end_time": "number"
}]`

You should only response with the json array

"""

user_prompt = f"""
    Here is the transcript of the video:
    {transcriptJSON}
"""

## Step 5 — Call the model and show the Shorts ideas

This sends the prompts to OpenAI and prints the JSON in Markdown.

This is the cell that spends API credits. After it finishes, you should see five ideas with timestamps you can use to jump back into the original video.

In [ ]:


response = client.chat.completions.create(
    model="gpt-5",
    messages=[
        {"role": "system", "content": system_prompt}, 
        {"role": "user", "content": user_prompt}
    ],
    response_format={"type": "json_object"},
)

shortsJSON = response.choices[0].message.content

# format the json to display in a readable format in a Markdown
shortsJSON = json.loads(shortsJSON)

markdown_content = f"""
# YouTube Shorts Ideas

```json
{json.dumps(shortsJSON, indent=4)}
```
"""

display(Markdown(markdown_content))
